<a href="https://colab.research.google.com/github/DanielHashmi/Homework_Python_Projects/blob/main/25%20Projects/Bulk_File_Renamer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from datetime import datetime

class BulkFileRenamer:
    def __init__(self, root):
        self.root = root
        self.root.title("Bulk File Renamer")
        self.root.geometry("600x500")
        self.root.resizable(True, True)

        self.folder_path = tk.StringVar()
        self.prefix = tk.StringVar()
        self.suffix = tk.StringVar()
        self.replace_text = tk.StringVar()
        self.replace_with = tk.StringVar()
        self.add_date = tk.BooleanVar(value=False)
        self.date_format = tk.StringVar(value="%Y%m%d")
        self.file_counter = tk.BooleanVar(value=False)
        self.counter_start = tk.IntVar(value=1)
        self.counter_digits = tk.IntVar(value=3)
        self.case_option = tk.StringVar(value="no_change")

        self.setup_ui()

    def setup_ui(self):
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.pack(fill=tk.BOTH, expand=True)

        folder_frame = ttk.LabelFrame(main_frame, text="Select Folder", padding="5")
        folder_frame.pack(fill=tk.X, pady=5)

        ttk.Entry(folder_frame, textvariable=self.folder_path, width=50).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
        ttk.Button(folder_frame, text="Browse", command=self.browse_folder).pack(side=tk.RIGHT, padx=5)

        options_frame = ttk.LabelFrame(main_frame, text="Renaming Options", padding="5")
        options_frame.pack(fill=tk.BOTH, expand=True, pady=5)

        ttk.Label(options_frame, text="Add Prefix:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=2)
        ttk.Entry(options_frame, textvariable=self.prefix, width=30).grid(row=0, column=1, sticky=tk.W, padx=5, pady=2)

        ttk.Label(options_frame, text="Add Suffix:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=2)
        ttk.Entry(options_frame, textvariable=self.suffix, width=30).grid(row=1, column=1, sticky=tk.W, padx=5, pady=2)

        ttk.Label(options_frame, text="Replace Text:").grid(row=2, column=0, sticky=tk.W, padx=5, pady=2)
        ttk.Entry(options_frame, textvariable=self.replace_text, width=30).grid(row=2, column=1, sticky=tk.W, padx=5, pady=2)

        ttk.Label(options_frame, text="Replace With:").grid(row=3, column=0, sticky=tk.W, padx=5, pady=2)
        ttk.Entry(options_frame, textvariable=self.replace_with, width=30).grid(row=3, column=1, sticky=tk.W, padx=5, pady=2)

        ttk.Label(options_frame, text="Change Case:").grid(row=4, column=0, sticky=tk.W, padx=5, pady=2)
        case_combo = ttk.Combobox(options_frame, textvariable=self.case_option, width=28, state="readonly")
        case_combo['values'] = ("no_change", "lowercase", "uppercase", "title_case")
        case_combo.current(0)
        case_combo.grid(row=4, column=1, sticky=tk.W, padx=5, pady=2)

        date_frame = ttk.Frame(options_frame)
        date_frame.grid(row=5, column=0, columnspan=2, sticky=tk.W, padx=5, pady=2)

        ttk.Checkbutton(date_frame, text="Add Date", variable=self.add_date).pack(side=tk.LEFT)
        ttk.Label(date_frame, text="Format:").pack(side=tk.LEFT, padx=(10, 5))
        ttk.Entry(date_frame, textvariable=self.date_format, width=10).pack(side=tk.LEFT)
        ttk.Label(date_frame, text="(e.g., %Y%m%d)").pack(side=tk.LEFT, padx=5)

        counter_frame = ttk.Frame(options_frame)
        counter_frame.grid(row=6, column=0, columnspan=2, sticky=tk.W, padx=5, pady=2)

        ttk.Checkbutton(counter_frame, text="Add Counter", variable=self.file_counter).pack(side=tk.LEFT)
        ttk.Label(counter_frame, text="Start:").pack(side=tk.LEFT, padx=(10, 5))
        ttk.Spinbox(counter_frame, from_=1, to=1000, width=5, textvariable=self.counter_start).pack(side=tk.LEFT)
        ttk.Label(counter_frame, text="Digits:").pack(side=tk.LEFT, padx=(10, 5))
        ttk.Spinbox(counter_frame, from_=1, to=10, width=5, textvariable=self.counter_digits).pack(side=tk.LEFT)

        button_frame = ttk.Frame(main_frame)
        button_frame.pack(fill=tk.X, pady=10)

        ttk.Button(button_frame, text="Preview Changes", command=self.preview_changes).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Rename Files", command=self.rename_files).pack(side=tk.RIGHT, padx=5)

        preview_frame = ttk.LabelFrame(main_frame, text="Preview", padding="5")
        preview_frame.pack(fill=tk.BOTH, expand=True, pady=5)

        self.preview_tree = ttk.Treeview(preview_frame, columns=("Original", "New"), show="headings")
        self.preview_tree.heading("Original", text="Original Filename")
        self.preview_tree.heading("New", text="New Filename")
        self.preview_tree.column("Original", width=250)
        self.preview_tree.column("New", width=250)
        self.preview_tree.pack(fill=tk.BOTH, expand=True)

        scrollbar = ttk.Scrollbar(preview_frame, orient=tk.VERTICAL, command=self.preview_tree.yview)
        self.preview_tree.configure(yscroll=scrollbar.set)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

    def browse_folder(self):
        folder_selected = filedialog.askdirectory()
        if folder_selected:
            self.folder_path.set(folder_selected)

    def get_new_filename(self, original_filename, counter=None):
        name, ext = os.path.splitext(original_filename)

        if self.replace_text.get():
            name = name.replace(self.replace_text.get(), self.replace_with.get())

        if self.case_option.get() == "lowercase":
            name = name.lower()
        elif self.case_option.get() == "uppercase":
            name = name.upper()
        elif self.case_option.get() == "title_case":
            name = name.title()

        new_name = name

        if self.prefix.get():
            new_name = f"{self.prefix.get()}{new_name}"

        if self.add_date.get():
            date_str = datetime.now().strftime(self.date_format.get())
            new_name = f"{new_name}_{date_str}"

        if self.file_counter.get() and counter is not None:
            counter_format = f"{{:0{self.counter_digits.get()}d}}"
            counter_str = counter_format.format(self.counter_start.get() + counter)
            new_name = f"{new_name}_{counter_str}"

        if self.suffix.get():
            new_name = f"{new_name}{self.suffix.get()}"

        return f"{new_name}{ext}"

    def preview_changes(self):
        folder = self.folder_path.get()
        if not folder or not os.path.isdir(folder):
            messagebox.showerror("Error", "Please select a valid folder")
            return

        for item in self.preview_tree.get_children():
            self.preview_tree.delete(item)

        try:
            files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]

            for i, file in enumerate(files):
                new_name = self.get_new_filename(file, counter=i)
                self.preview_tree.insert("", tk.END, values=(file, new_name))

            if not files:
                messagebox.showinfo("Info", "No files found in the selected directory")
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred: {str(e)}")

    def rename_files(self):
        folder = self.folder_path.get()
        if not folder or not os.path.isdir(folder):
            messagebox.showerror("Error", "Please select a valid folder")
            return

        try:
            files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
            renamed_count = 0
            skipped_count = 0

            for i, file in enumerate(files):
                original_path = os.path.join(folder, file)
                new_name = self.get_new_filename(file, counter=i)
                new_path = os.path.join(folder, new_name)

                if new_path == original_path:
                    skipped_count += 1
                    continue

                if os.path.exists(new_path):
                    skipped_count += 1
                    continue

                os.rename(original_path, new_path)
                renamed_count += 1

            messagebox.showinfo("Success", f"Renamed {renamed_count} files. Skipped {skipped_count} files.")
            self.preview_changes()

        except Exception as e:
            messagebox.showerror("Error", f"An error occurred: {str(e)}")

if __name__ == "__main__":
    root = tk.Tk()
    app = BulkFileRenamer(root)
    root.mainloop()